# 05 — Cohort Analysis
Monthly cohort retention, revenue per cohort, LTV curves, and avg orders per cohort.

**Prerequisite:** Run `01_data_generation.ipynb` first.

In [ ]:
# Cell 1 — Imports & Connection
import sqlite3, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

BASE_DIR = Path('..').resolve()
DB_PATH  = BASE_DIR / 'data' / 'raw' / 'ecommerce.db'
assert DB_PATH.exists(), 'Run notebook 01 first'
conn = sqlite3.connect(DB_PATH)
print('Connected.')

In [ ]:
# Cell 2 — Build Cohort Base Table in Python
df_orders = pd.read_sql(
    "SELECT user_id, order_date FROM orders WHERE status='completed'", conn,
    parse_dates=['order_date']
)

# First purchase month per user = cohort
df_orders['order_month'] = df_orders['order_date'].dt.to_period('M')
cohort_map = df_orders.groupby('user_id')['order_month'].min().rename('cohort_month')
df_orders  = df_orders.join(cohort_map, on='user_id')

# Month index (0 = acquisition month)
df_orders['month_number'] = (
    (df_orders['order_month'].dt.year  - df_orders['cohort_month'].dt.year)  * 12 +
    (df_orders['order_month'].dt.month - df_orders['cohort_month'].dt.month)
)

print(f'Cohorts found  : {df_orders["cohort_month"].nunique()}')
print(f'Month range    : 0 – {df_orders["month_number"].max()}')
print(df_orders.head(3).to_string())

In [ ]:
# Cell 3 — Cohort Retention Heatmap
cohort_pivot = (
    df_orders
    .groupby(['cohort_month','month_number'])['user_id'].nunique()
    .unstack()
)

cohort_sizes = cohort_pivot.iloc[:, 0]          # month 0 = cohort size
retention    = cohort_pivot.divide(cohort_sizes, axis=0) * 100
retention    = retention.round(1)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    retention,
    annot=True, fmt='.0f', cmap='YlOrRd_r',
    linewidths=0.3, linecolor='white',
    vmin=0, vmax=100,
    ax=ax, cbar_kws={'label': 'Retention %'}
)
ax.set_title('Monthly Cohort Retention Heatmap (% of cohort purchasing each month)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort Month')
ax.set_yticklabels([str(p) for p in retention.index], rotation=0)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'cohort_retention_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Retention Curves per Cohort
fig, ax = plt.subplots(figsize=(13, 5))

colors = sns.color_palette('tab10', len(retention))
for idx, (cohort, row) in enumerate(retention.iterrows()):
    valid = row.dropna()
    ax.plot(valid.index, valid.values, marker='o', markersize=4,
            label=str(cohort), color=colors[idx], linewidth=1.5)

ax.set_title('Cohort Retention Curves', fontsize=13, fontweight='bold')
ax.set_xlabel('Month Number')
ax.set_ylabel('Retention %')
ax.legend(title='Cohort', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'cohort_retention_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Cohort Revenue Over Time
df_rev = pd.read_sql(
    "SELECT user_id, order_date, total_amount FROM orders WHERE status='completed'",
    conn, parse_dates=['order_date']
)
df_rev['order_month']  = df_rev['order_date'].dt.to_period('M')
df_rev = df_rev.join(cohort_map, on='user_id')
df_rev['month_number'] = (
    (df_rev['order_month'].dt.year  - df_rev['cohort_month'].dt.year)  * 12 +
    (df_rev['order_month'].dt.month - df_rev['cohort_month'].dt.month)
)

cohort_revenue = (
    df_rev
    .groupby(['cohort_month','month_number'])['total_amount'].sum()
    .unstack()
    .round(0)
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    cohort_revenue / 1000,   # show in $K
    annot=True, fmt='.0f', cmap='Blues',
    linewidths=0.3, linecolor='white',
    ax=ax, cbar_kws={'label': 'Revenue ($K)'}
)
ax.set_title('Cohort Revenue Heatmap ($K per month)', fontsize=12, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort Month')
ax.set_yticklabels([str(p) for p in cohort_revenue.index], rotation=0)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'cohort_revenue_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Cumulative LTV Curves
cumulative_rev = cohort_revenue.cumsum(axis=1)

# Normalise by cohort size to get per-user LTV
ltv = cumulative_rev.divide(cohort_sizes, axis=0).round(2)

fig, ax = plt.subplots(figsize=(13, 5))
colors = sns.color_palette('tab10', len(ltv))
for idx, (cohort, row) in enumerate(ltv.iterrows()):
    valid = row.dropna()
    ax.plot(valid.index, valid.values, marker='o', markersize=4,
            label=str(cohort), color=colors[idx], linewidth=1.8)

ax.set_title('Cumulative LTV per User by Cohort', fontsize=13, fontweight='bold')
ax.set_xlabel('Month Number')
ax.set_ylabel('Cumulative Revenue per User ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(title='Cohort', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'cohort_ltv_curves.png', bbox_inches='tight')
plt.show()

print('Max observed LTV per cohort (last available month):')
for cohort, row in ltv.iterrows():
    print(f'  {str(cohort)}  ${row.dropna().iloc[-1]:,.2f}')

In [ ]:
# Cell 7 — Cohort Size & Avg Orders per User
cohort_stats = (
    df_orders
    .groupby(['cohort_month','user_id'])['order_month']
    .count()
    .reset_index()
    .rename(columns={'order_month': 'order_count'})
    .groupby('cohort_month')
    .agg(
        cohort_size    = ('user_id',     'count'),
        total_orders   = ('order_count', 'sum'),
        avg_orders     = ('order_count', 'mean')
    )
    .reset_index()
)
cohort_stats['cohort_month'] = cohort_stats['cohort_month'].astype(str)
cohort_stats['avg_orders']   = cohort_stats['avg_orders'].round(2)
print(cohort_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Cohort Size & Engagement', fontsize=12, fontweight='bold')

axes[0].bar(cohort_stats['cohort_month'], cohort_stats['cohort_size'], color='steelblue')
axes[0].set_title('Cohort Size (new buyers per month)')
axes[0].set_ylabel('Users')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(cohort_stats['cohort_month'], cohort_stats['avg_orders'], color='darkorange')
axes[1].set_title('Avg Orders per User per Cohort')
axes[1].set_ylabel('Avg Orders')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(cohort_stats['avg_orders']):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'processed' / 'cohort_size_engagement.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8 — Plotly Interactive Retention Heatmap
ret_reset = retention.copy()
ret_reset.index = [str(i) for i in ret_reset.index]
ret_reset.columns = [f'M{c}' for c in ret_reset.columns]

fig = px.imshow(
    ret_reset,
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=100,
    text_auto='.0f',
    title='Interactive Cohort Retention Heatmap (%)',
    labels=dict(x='Month Number', y='Cohort', color='Retention %')
)
fig.update_layout(height=420)
fig.show()

conn.close()
print('Done. All cohort charts saved.')

---
**All notebooks complete.**

Next step: Open `tableau/dashboard_notes.md` to build the Tableau dashboard using the CSVs in `data/processed/`.